# CadQuery para Arquitetos
## Notebook 2 — Sólidos Primitivos e Operações Booleanas

---

Neste notebook vamos explorar em profundidade as **formas primitivas** do CadQuery e as **operações booleanas**, que permitem combinar sólidos simples para criar geometrias arquitetônicas mais complexas.

---

## Importações

In [ ]:
import cadquery as cq
from ocp_vscode import show, set_port, show_object

In [ ]:
set_port(3939)

---

## Sólidos Primitivos

Os **sólidos primitivos** são formas geométricas básicas que o CadQuery consegue criar diretamente, sem necessidade de construção passo a passo. São o vocabulário fundamental da modelagem paramétrica.

Nos exemplos a seguir, cada primitiva é apresentada com seus parâmetros detalhados e um exemplo de uso com contexto arquitetônico.

---

### Caixa — `.box()`

A caixa retangular é a primitiva mais usada em arquitetura. Representa qualquer volume ortogonal: paredes, lajes, pilares retangulares, blocos.

```python
.box(length, width, height)
```

| Parâmetro | Eixo | Descrição |
|-----------|------|-----------|
| `length` | X | Comprimento |
| `width`  | Y | Largura |
| `height` | Z | Altura |

Por padrão, a caixa é criada **centrada na origem** do Workplane. O parâmetro `centered` controla esse comportamento:
- `centered=True` → centro geométrico na origem (padrão)
- `centered=False` → canto inferior esquerdo na origem

In [ ]:
# Caixa centrada na origem (padrão)
pilar_centrado = cq.Workplane("XY").box(0.4, 0.4, 3.0)

# Caixa com canto inferior esquerdo na origem
# Mais intuitivo para posicionamento em plantas baixas
pilar_canto = cq.Workplane("XY").box(0.4, 0.4, 3.0, centered=False)

show(
    pilar_centrado,
    pilar_canto,
    names=["Centrado", "Canto na origem"]
)

---

### Cilindro — `.cylinder()`

O cilindro representa pilares circulares, colunas, tanques, torres cilíndricas.

```python
.cylinder(height, radius, angle=360)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `height` | Altura |
| `radius` | Raio da base |
| `angle`  | Ângulo de abertura em graus — padrão `360` (cilindro completo) |

O parâmetro `angle` permite criar **segmentos de cilindro**, úteis para curvas parciais e elementos curvos.

In [ ]:
# Coluna circular completa
coluna = cq.Workplane("XY").cylinder(3.0, 0.2)

# Parede curva: segmento de cilindro de 90°
parede_curva = cq.Workplane("XY").cylinder(2.8, 1.5, angle=90)

# Parede curva em semicírculo: 180°
parede_semi = cq.Workplane("XY").cylinder(2.8, 2.5, angle=180)

show(
    coluna,
    parede_curva,
    parede_semi,
    names=["Coluna", "Parede curva 90°", "Parede semicircular"]
)

---

### Esfera — `.sphere()`

A esfera é usada em cúpulas, elementos decorativos, volumes orgânicos e estudo de formas.

```python
.sphere(radius, angle1=-90, angle2=90, angle3=360)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `radius`  | Raio |
| `angle1`  | Ângulo inicial na vertical (latitude inferior) — padrão `-90°` |
| `angle2`  | Ângulo final na vertical (latitude superior) — padrão `90°` |
| `angle3`  | Ângulo de abertura horizontal — padrão `360°` |

Controlando `angle1` e `angle2` é possível criar **calotas esféricas** — como cúpulas.

In [ ]:
# Esfera completa
esfera = cq.Workplane("XY").sphere(2.0)

# Calota superior: metade de cima da esfera
# angle1=0 corta na equador, angle2=90 vai até o topo
cupula = cq.Workplane("XY").sphere(2.0, angle1=0, angle2=90)

# Calota rasa: apenas o terço superior
cupula_rasa = cq.Workplane("XY").sphere(2.0, angle1=60, angle2=90)

show(
    esfera,
    cupula,
    cupula_rasa,
    names=["Esfera completa", "Cúpula", "Cúpula rasa"]
)

---

### Cone e Tronco de Cone — `.cone()`

O cone representa telhados pontiagudos, pirâmides, elementos de cobertura e volumes troncocônicos.

```python
.cone(height, radius1, radius2)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `height`  | Altura |
| `radius1` | Raio da base inferior |
| `radius2` | Raio da base superior — `0` = cone pontiagudo |

Quando `radius2 > 0`, o resultado é um **tronco de cone**.

In [ ]:
# Cone pontiagudo — telhado cônico
telhado_conico = cq.Workplane("XY").cone(2.0, 2.5, 0)

# Tronco de cone — base maior em baixo
tronco = cq.Workplane("XY").cone(3.0, 3.0, 1.5)

# Tronco invertido — base menor em baixo
# (útil para marquises e balanços)
tronco_invertido = cq.Workplane("XY").cone(2.0, 1.0, 3.0)

show(
    telhado_conico,
    tronco,
    tronco_invertido,
    names=["Telhado cônico", "Tronco de cone", "Tronco invertido"]
)

---

### Toro — `.torus()`

O toro é a forma geométrica de uma rosca ou câmara de ar. Em arquitetura, aparece em molduras curvas, elementos decorativos toroidais e coberturas anulares.

```python
.torus(radius1, radius2)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `radius1` | Raio do anel central (distância do centro à seção tubular) |
| `radius2` | Raio do tubo (espessura do anel) |

In [ ]:
# Toro simples — como uma argola
anel = cq.Workplane("XY").torus(3.0, 0.5)

# Toro com tubo mais espesso
anel_grosso = cq.Workplane("XY").torus(3.0, 1.2)

show(
    anel,
    anel_grosso,
    names=["Anel fino", "Anel grosso"]
)

---

### Cunha — `.wedge()`

A cunha é um volume prismático triangular. É a primitiva mais próxima de um telhado de duas águas ou de elementos em rampa.

```python
.wedge(dx, dy, dz, xmin, zmin, xmax, zmax)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `dx` | Comprimento total no eixo X |
| `dy` | Profundidade no eixo Y |
| `dz` | Altura total no eixo Z |
| `xmin`, `zmin` | Canto inferior esquerdo do topo |
| `xmax`, `zmax` | Canto superior direito do topo |

Para criar um **telhado de duas águas simples**, o topo é uma linha no centro: `xmin=xmax=dx/2` e `zmin=0`, `zmax=dz`.

In [ ]:
# Cunha simples — rampa
rampa = cq.Workplane("XY").wedge(5.0, 3.0, 1.5, 0, 0, 0, 1.5)

# Telhado de duas águas
# Base: 8m x 6m, cumeeira a 2m de altura no centro
dx = 8.0
dy = 6.0
dz = 2.0

telhado = cq.Workplane("XY").wedge(dx, dy, dz, dx/2, 0, dx/2, dz)

show(
    rampa,
    telhado,
    names=["Rampa", "Telhado duas águas"]
)

---

## Operações Booleanas

As **operações booleanas** permitem combinar dois ou mais sólidos para criar formas que não seriam possíveis com uma única primitiva. São as operações mais importantes da modelagem de sólidos.

Existem três operações booleanas:

| Operação | Método | Resultado |
|----------|--------|-----------|
| União | `.union(outro)` | Une os dois sólidos em um único volume |
| Subtração | `.cut(outro)` | Remove o volume do segundo sólido do primeiro |
| Interseção | `.intersect(outro)` | Mantém apenas o volume compartilhado pelos dois |

> 💡 **Dica**: nas operações booleanas, a **ordem importa** para subtração e interseção. `a.cut(b)` é diferente de `b.cut(a)`.

---

### União — `.union()`

A união **funde** dois sólidos em um único objeto. As superfícies internas de contato são eliminadas, resultando em um volume contínuo.

Contexto arquitetônico: unir o corpo de um edifício com uma marquise, uma escada com um patamar, ou dois volumes de uma composição.

In [ ]:
# Corpo principal do edifício
corpo = cq.Workplane("XY").box(10.0, 8.0, 6.0)

# Volume de acesso — conectado ao corpo principal
acesso = cq.Workplane("XY", origin=(7.0, 0, -1.5)).box(4.0, 4.0, 3.0)

# Antes da união: os dois volumes separados
show(
    corpo,
    acesso,
    names=["Corpo", "Acesso"]
)

In [ ]:
# Após a união: um único sólido contínuo
edificio = corpo.union(acesso)

show(edificio, names=["Edifício unido"])

---

### Subtração — `.cut()`

A subtração **remove** do primeiro sólido o volume ocupado pelo segundo. O sólido subtraído funciona como um "molde" negativo.

Contexto arquitetônico: abrir vãos em paredes, criar janelas e portas, escavar terraços, abrir pátios em edifícios.

In [ ]:
# Parede sólida
parede = cq.Workplane("XY").box(8.0, 0.3, 3.0)

# Vão da janela — posicionado no centro da parede
vao_janela = cq.Workplane("XY", origin=(0, 0, 0.4)).box(1.5, 0.5, 1.5)

# Vão da porta — posicionado à esquerda
vao_porta = cq.Workplane("XY", origin=(-3.0, 0, -0.5)).box(1.0, 0.5, 2.0)

# Abrindo os vãos na parede com duas subtrações encadeadas
parede_com_vaos = parede.cut(vao_janela).cut(vao_porta)

show(parede_com_vaos, names=["Parede com vãos"])

In [ ]:
# Exemplo 2: pátio escavado em um volume compacto

# Volume total do edifício
bloco = cq.Workplane("XY").box(12.0, 12.0, 4.0)

# Pátio interno — um volume subtraído do centro
patio = cq.Workplane("XY", origin=(0, 0, 0.5)).box(6.0, 6.0, 5.0)

edificio_patio = bloco.cut(patio)

show(edificio_patio, names=["Edifício com pátio"])

---

### Interseção — `.intersect()`

A interseção **mantém apenas o volume que os dois sólidos compartilham**. O resultado é a região de sobreposição entre eles.

Contexto arquitetônico: encontrar o volume máximo que cabe simultaneamente dentro de duas restrições, criar formas a partir do cruzamento de geometrias, estudar interferências entre sistemas.

In [ ]:
# Exemplo: cruzamento de dois volumes cilíndricos
# Útil para estudar formas resultantes da intersecção de estruturas curvas

cilindro_h = cq.Workplane("XZ").cylinder(6.0, 2.0)
cilindro_v = cq.Workplane("XY").cylinder(6.0, 2.0)

# Antes da interseção
show(
    cilindro_h,
    cilindro_v,
    names=["Cilindro horizontal", "Cilindro vertical"]
)

In [ ]:
# Resultado da interseção: apenas o volume compartilhado
intersecao = cilindro_h.intersect(cilindro_v)

show(intersecao, names=["Interseção"])

In [ ]:
# Exemplo arquitetônico: volume permitido pelo cruzamento de
# um envelope de altura máxima (caixão) e um cone de visibilidade

# Envelope cúbico: volume máximo permitido pela legislação
envelope = cq.Workplane("XY").box(10.0, 10.0, 8.0)

# Cone de afastamento: restrição de ângulo a partir da rua
cone_afastamento = cq.Workplane("XY", origin=(0, -7, -4)).cone(12.0, 14.0, 2.0)

volume_resultante = envelope.intersect(cone_afastamento)

show(
    envelope,
    cone_afastamento,
    names=["Envelope", "Cone de afastamento"]
)

In [ ]:
show(volume_resultante, names=["Volume resultante"])

---

## Combinando Operações Booleanas

Na prática, projetos arquitetônicos combinam várias operações booleanas em sequência. Veja um exemplo de uma composição mais elaborada: um edifício com torre, cobertura cônica e aberturas.

In [ ]:
# --- Base do edifício ---
base_larg  = 10.0
base_prof  = 8.0
base_alt   = 4.0
base = cq.Workplane("XY").box(base_larg, base_prof, base_alt)

# --- Torre central ---
torre_raio = 1.5
torre_alt  = 6.0
torre = cq.Workplane("XY").cylinder(torre_alt, torre_raio)

# --- Cobertura cônica da torre ---
cobertura = cq.Workplane("XY", origin=(0, 0, torre_alt / 2)).cone(2.5, 2.0, 0)

# --- Vãos das janelas ---
janela_frente = cq.Workplane("XY", origin=(0, 0, 0.5)).box(2.0, base_prof + 0.5, 1.5)
janela_lado   = cq.Workplane("XY", origin=(0, 0, 0.5)).box(base_larg + 0.5, 2.0, 1.5)

# --- Montagem ---
# 1. Unir base com torre
edificio = base.union(torre)

# 2. Adicionar cobertura cônica
edificio = edificio.union(cobertura)

# 3. Abrir vãos das janelas
edificio = edificio.cut(janela_frente)
edificio = edificio.cut(janela_lado)

show(edificio, names=["Composição final"])

---

## Exercício

Crie um modelo volumétrico de um **pavilhão simples** com os seguintes elementos:

1. **Base**: volume retangular representando as paredes
2. **Cobertura**: use a cunha (`.wedge()`) para criar um telhado de duas águas
3. **Vãos**: abra pelo menos duas janelas e uma porta usando subtração
4. **Detalhe**: adicione colunas cilíndricas em pelo menos dois cantos usando união

Use variáveis para todas as dimensões e exiba o resultado final com `show()`.

In [ ]:
# Escreva seu código aqui



---

## Resumo

Neste notebook você aprendeu:

- As seis primitivas do CadQuery: `box`, `cylinder`, `sphere`, `cone`, `torus`, `wedge`
- O parâmetro `centered` da caixa e o parâmetro `angle` do cilindro e da esfera
- Como criar **cúpulas** com `sphere(angle1, angle2)`
- Como criar **telhados** e **rampas** com `wedge()`
- As três **operações booleanas**: `.union()`, `.cut()`, `.intersect()`
- Como **encadear** operações booleanas para construir modelos complexos

No próximo notebook veremos como trabalhar com **perfis 2D e extrusões** — a base da modelagem arquitetônica a partir de plantas baixas.

---
*CadQuery para Arquitetos — Notebook 2*